In [32]:
import pandas as pd
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
df = pd.read_parquet('../data/processed/dblp_clustered.parquet')
df_rag = df.groupby('cluster_label', group_keys=False).apply(
    lambda x: x.sample(frac=0.25, random_state=42)
)
print(df_rag.shape)
print(df_rag.columns.tolist())

(49999, 10)
['type', 'year', 'title', 'authors', 'source', 'author_count', 'title_length', 'first_author', 'cluster', 'cluster_name']
<class 'pandas.DataFrame'>
Index: 49999 entries, 19835 to 174161
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   type          49999 non-null  str  
 1   year          49999 non-null  int64
 2   title         49999 non-null  str  
 3   authors       49999 non-null  str  
 4   source        49999 non-null  str  
 5   author_count  49999 non-null  int64
 6   title_length  49999 non-null  int64
 7   first_author  49999 non-null  str  
 8   cluster       49999 non-null  int32
 9   cluster_name  49999 non-null  str  
dtypes: int32(1), int64(3), str(6)
memory usage: 13.7 MB
None


In [17]:
df_rag['cluster'].value_counts()

cluster
12    9315
2     5159
15    4283
0     3223
1     3192
13    2224
6     2120
5     2060
4     1982
9     1903
18    1860
8     1846
11    1797
7     1622
14    1414
16    1308
10    1230
19    1222
17    1219
3     1020
Name: count, dtype: int64

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model downloaded, embedding dimension:  {model.get_sentence_embedding_dimension()}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1534.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model downloaded, embedding dimension:  384


In [18]:
embeddings = model.encode(
    df_rag['title'].tolist(),
    batch_size=512,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f'Embeddings shape: {embeddings.shape}')

Batches: 100%|██████████| 98/98 [03:01<00:00,  1.85s/it]

Embeddings shape: (49999, 384)


In [19]:
np.save('../data/processed/embeddings.npy', embeddings)

In [20]:
embeddings = np.load('../data/processed/embeddings.npy')
print(f'Shape: {embeddings.shape}')
print(f'Dtype: {embeddings.dtype}')
print(f'Example vector (first five values) : {embeddings[0][:5]}')

Shape: (49999, 384)
Dtype: float32
Example vector (first five values) : [ 0.07357834  0.03428687 -0.04251426 -0.03234662 -0.00040755]


In [23]:
client = chromadb.PersistentClient(path='../data/chromadb')

collection = client.get_or_create_collection(
    name='articles',
    metadata={'hnsw:space': 'cosine'}
)
print(f'Collection create, docs inside: {collection.count()}')

Collection create, docs inside: 0


In [24]:
batch_size = 5000

for i in range(0, len(df_rag), batch_size):
    batch = df_rag.iloc[i:i+batch_size]
    collection.add(
        ids=[str(idx) for idx in batch.index],
        embeddings=embeddings[i:i+batch_size].tolist(),
        documents=batch['title'].tolist(),
        metadatas=[{
            'year': int(row['year']),
            'cluster' : int(row['cluster']), 
            'cluster_name' : row['cluster_name'],
            'authors' : row['authors'],
            'type' : row['type'], 
            'source' : row['source']
        } for _, row in batch.iterrows()]
    ) 
    
print(f'Docs inside ChromaDB: {collection.count()}')

Docs inside ChromaDB: 49999


In [28]:
test_queries = [
    "deep learning image classification",
    "natural language processing transformers",
    "graph neural networks",
    "quantum computing algorithms",
    "reinforcement learning robotics"
]

for query in test_queries:
    query_embedding = model.encode([query], convert_to_numpy=True)
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=3
    )
    print(f"\n Zapytanie: '{query}'")
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        print(f"  {i+1}. {doc}")
        print(f"     Klaster: {meta['cluster_name']} | Rok: {meta['year']} | Score: {1-dist:.3f}")


 Zapytanie: 'deep learning image classification'
  1. Image Classification of Lung X-ray Images using Deep learning.
     Klaster: Deep Learning | Rok: 2022 | Score: 0.620
  2. Stress and Adaptation: Applying Anna Karenina Principle in Deep Learning for Image Classification.
     Klaster: Deep Learning | Rok: 2023 | Score: 0.616
  3. Classification of Indoor-Outdoor Scene Using Deep Learning Techniques.
     Klaster: Deep Learning | Rok: 2021 | Score: 0.614

 Zapytanie: 'natural language processing transformers'
  1. Semantic Correspondence with Transformers.
     Klaster: Computer Vision | Rok: 2021 | Score: 0.648
  2. Generative Pretrained Structured Transformers: Unsupervised Syntactic Language Models at Scale.
     Klaster: LLMs & Foundation Models | Rok: 2024 | Score: 0.642
  3. On Robustness of Finetuned Transformer-based NLP Models.
     Klaster: LLMs & Foundation Models | Rok: 2023 | Score: 0.632

 Zapytanie: 'graph neural networks'
  1. Continuous Graph Neural Networks.
     

In [33]:
load_dotenv('../.env')

llm = ChatGroq(
    model='llama-3.1-8b-instant',
    api_key=os.getenv('GROQ_API_KEY'),
    temperature=0.1
)

prompt = ChatPromptTemplate.from_template("""
You are a research assistant helping to find relevant academic papers.
Based on the following papers from a database, answer the user's question.
Always base your answer ONLY on the provided papers, not on your general knowledge.

Relevant papers:
{context}

User question: {question}

Provide a helpful answer referencing the specific papers above.
""")

test_response = llm.invoke("Say 'RAG pipeline working!' in one sentence.")
print(test_response.content)

The RAG pipeline is successfully operational, efficiently processing and delivering data.


In [40]:
def rag_query(question, n : int = 5):
    query_embedding = model.encode([question], convert_to_numpy=True)
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n
    )
    context = ''
    for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        context += f"{i+1}. \"{doc}\" — {meta['authors']} ({meta['year']}) | Topic: {meta['cluster_name']}\n"
    chain = prompt | llm 
    response = chain.invoke({
        'context' : context,
        'question' : question
    })
    return response.content, context

In [41]:
question = "What are the recent advances in graph neural networks?"
answer, context = rag_query(question)

print('Znalezione papiery:')
print(context)
print('Odpowiedz LLM:')
print(answer)

Znalezione papiery:
1. "Graph Neural Networks with High-order Feature Interactions." — Kaize Ding, Yichuan Li 0001, Jundong Li, Chenghao Liu, Huan Liu 0001 (2019) | Topic: Graph & Wireless Networks
2. "mGNN: Generalizing the Graph Neural Networks to the Multilayer Case." — Marco Grassia, Manlio De Domenico, Giuseppe Mangioni (2021) | Topic: Graph & Wireless Networks
3. "Continuous Graph Neural Networks." — Louis-Pascal A. C. Xhonneux, Meng Qu, Jian Tang 0005 (2020) | Topic: Graph & Wireless Networks
4. "Topological Graph Neural Networks." — Max Horn, Edward De Brouwer, Michael Moor, Yves Moreau, Bastian Rieck, Karsten M. Borgwardt (2022) | Topic: Graph & Wireless Networks
5. "Random Features Strengthen Graph Neural Networks." — Ryoma Sato, Makoto Yamada, Hisashi Kashima (2020) | Topic: Graph & Wireless Networks

Odpowiedz LLM:
Based on the provided papers, recent advances in graph neural networks include:

1. **High-order feature interactions**: The paper "Graph Neural Networks with Hi

In [42]:
question = "Tell me something about decision trees?"
answer, context = rag_query(question)

print('Znalezione papiery:')
print(context)
print('Odpowiedz LLM:')
print(answer)

Znalezione papiery:
1. "Decision trees as partitioning machines to characterize their generalization properties." — Jean-Samuel Leboeuf, Frédéric Leblanc, Mario Marchand (2020) | Topic: AI & Knowledge Systems
2. "GATree: Evolutionary decision tree classifier in Python." — Tadej Lahovnik, Saso Karakatic (2024) | Topic: AI & Knowledge Systems
3. "Towards a Comprehensive Evaluation of Decision Rules and Decision Mining Algorithms Beyond Accuracy." — Beate Wais, Stefanie Rinderle-Ma (2024) | Topic: AI & Knowledge Systems
4. "Training decision trees as replacement for convolution layers." — Wolfgang Fuhl, Gjergji Kasneci, Wolfgang Rosenstiel, Enkelejda Kasneci (2019) | Topic: AI & Knowledge Systems
5. "Safety Verification of Decision-Tree Policies in Continuous Time." — Christian Schilling 0001, Anna Lukina, Emir Demirovic, Kim Guldstrand Larsen (2023) | Topic: AI & Knowledge Systems

Odpowiedz LLM:
Based on the provided papers, here's what I can tell you about decision trees:

Decision tre